<a href="https://colab.research.google.com/github/vssreemann-cloud/CALCULATOR/blob/main/Copy_of_recipe_agent_gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# About
This colab is an introduction to a RAG based ReAct agent that looks at a recipe book, reasons the best course of action and generates a response.

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to aistudio.google.com
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0



In [1]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

API key loaded from Colab Secrets.


In [2]:
# Quiet ensures your UI is not full of logs.
!pip install google-generativeai --quiet
print("Done.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-spanner 3.71.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-storage-control 1.17.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-language 2.22.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-translate 3.28.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-bigquery-storage 2.43.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
tensorflow 2.21.0 requires protobuf<8.0.0,>=6.31.1, but you have protobuf 5.29.6 which is incompatible.
google-cloud-functions 1.25.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
grpc-google-iam

In [8]:
# @title Create a database of recipes.
RECIPES = {
    "chicken_biryani": (
        "500g chicken pieces",
        "2 cups basmati rice",
        "1 cup plain yogurt",
        "2 large onions thinly sliced",
        "2 tomatoes chopped",
        "2 tbsp ginger garlic paste",
        "2 green chilies slit",
        "1/2 cup chopped fresh mint leaves",
        "1/2 cup chopped fresh coriander leaves",
        "1 tsp turmeric powder",
        "2 tsp red chili powder",
        "2 tsp biryani masala",
        "1 tsp garam masala",
        "4 tbsp cooking oil or ghee",
        "Whole spices (bay leaf, cinnamon, cloves, cardamom)",
        "Salt to taste",
        "Soak basmati rice in water for 30 minutes. Drain and keep aside.",
        "In a large bowl combine chicken, yogurt, ginger garlic paste, turmeric, chili powder, biryani masala and salt. Mix well and marinate for 30 minutes.",
        "Heat oil or ghee in a large pot. Add whole spices and sliced onions. Fry until golden brown.",
        "Add chopped tomatoes and green chilies. Cook until tomatoes become soft.",
        "Add marinated chicken and cook for 10-15 minutes until partially cooked.",
        "Add mint leaves, coriander leaves and garam masala. Mix well.",
        "In another pot boil water with salt. Add soaked rice and cook until approximately 70 percent done. Drain the rice.",
        "Spread the partially cooked rice evenly over the chicken mixture.",
        "Cover tightly and cook on low heat for 20-25 minutes until the chicken and rice are completely cooked.",
        "Gently mix and serve hot with raita or salad."
    ),

    "chocolate_brownies": (
        "1/2 cup unsalted butter",
        "1 cup granulated sugar",
        "2 large eggs",
        "1 tsp vanilla extract",
        "1/3 cup unsweetened cocoa powder",
        "1/2 cup all-purpose flour",
        "1/4 tsp salt",
        "1/4 tsp baking powder",
        "1/2 cup chocolate chips",
        "Preheat oven to 350°F (175°C). Grease an 8-inch square baking pan or line it with parchment paper.",
        "Melt butter in a saucepan over low heat and allow it to cool slightly.",
        "In a large bowl combine melted butter and sugar. Mix until smooth.",
        "Add eggs and vanilla extract. Whisk until well combined.",
        "Sift cocoa powder, flour, salt and baking powder into the wet mixture.",
        "Gently fold the dry ingredients into the wet mixture until just combined. Do not overmix.",
        "Fold in chocolate chips evenly.",
        "Pour the batter into the prepared baking pan and spread evenly.",
        "Bake for 20-25 minutes or until a toothpick inserted near the center comes out with a few moist crumbs.",
        "Allow brownies to cool completely before cutting into squares.",
        "Serve as a dessert or with a scoop of vanilla ice cream."
    ),

    "vegetable_noodles": (
        "200g noodles",
        "1/2 cup thinly sliced cabbage",
        "1/2 cup julienned carrots",
        "1/2 cup sliced capsicum",
        "1/2 cup sliced onion",
        "2 cloves garlic minced",
        "1 tbsp soy sauce",
        "1 tbsp tomato ketchup",
        "1 tsp chili sauce",
        "1 tsp vinegar",
        "2 tbsp vegetable oil",
        "1/2 tsp black pepper powder",
        "Salt to taste",
        "Spring onions chopped for garnish",
        "Bring a large pot of water to a boil. Add noodles and cook according to package instructions until just tender.",
        "Drain the noodles and rinse briefly with cool water to prevent sticking.",
        "Heat vegetable oil in a large wok or skillet over medium-high heat.",
        "Add minced garlic and sliced onion. Stir-fry for 1-2 minutes.",
        "Add cabbage, carrots and capsicum. Cook on high heat for 3-4 minutes while stirring continuously.",
        "Add soy sauce, tomato ketchup, chili sauce and vinegar. Mix everything well.",
        "Add the cooked noodles to the wok and toss gently until evenly coated.",
        "Season with black pepper and salt to taste.",
        "Stir-fry for another 2-3 minutes until everything is heated through.",
        "Garnish with chopped spring onions and serve hot."
    ),

    "paneer_butter_masala": (
        "250g paneer cut into cubes",
        "2 medium onions roughly chopped",
        "3 medium tomatoes chopped",
        "2 tbsp butter",
        "1 tbsp cooking oil",
        "1 tbsp ginger garlic paste",
        "10 cashew nuts",
        "1/2 cup fresh cream",
        "1 tsp cumin seeds",
        "1 tsp red chili powder",
        "1/2 tsp turmeric powder",
        "1 tsp garam masala",
        "1 tsp coriander powder",
        "1 tsp kasuri methi",
        "Salt to taste",
        "Fresh coriander leaves for garnish",
        "Heat oil and butter in a pan over medium heat.",
        "Add cumin seeds and allow them to crackle.",
        "Add chopped onions and cook until soft and translucent.",
        "Add ginger garlic paste and cook for another minute.",
        "Add chopped tomatoes and cashew nuts. Cook until the tomatoes become soft.",
        "Allow the mixture to cool slightly and blend into a smooth paste.",
        "Heat the remaining butter in the pan and add red chili powder, turmeric powder and coriander powder.",
        "Add the prepared tomato-onion paste and cook for 5-7 minutes.",
        "Add garam masala, salt and a little water. Mix well and bring to a gentle simmer.",
        "Add paneer cubes and cook for 3-4 minutes.",
        "Stir in fresh cream and crushed kasuri methi. Simmer for another 2 minutes.",
        "Garnish with fresh coriander leaves and serve hot with naan, roti or rice."
    )
}

In [19]:
def suggest_substitutions(ingredient: str) -> str:
    """Suggest common culinary substitutes for a given ingredient."""
    substitutes = {
        "butter": "Equal parts coconut oil, vegan butter, or applesauce (for baking).",
        "termilk": "1 cup milk plus 1 tablespoon lemon juice or white vinegar (let sit for 5 minutes).",
        "sugar": "Honey, maple syrup, or jaggery (adjust liquid ratios accordingly).",
        "eggs": "1/4 cup unsweetened applesauce, mashed banana, or a flax egg (1 tbsp ground flaxseed + 3 tbsp water) per egg."
    }

    query = ingredient.lower().strip()
    for item, sub in substitutes.items():
        if item in query:
            return f"Suggested substitute for {ingredient}: {sub}"

    return f"No specific substitution found for '{ingredient}' in the quick reference database."

In [13]:
# @title Define a tool
def search_recipes(query: str) -> str:
    """Search the cookbook collection for recipes relevant to the query."""
    query_lower = query.lower()
    results = []
    for doc_name, content_tuple in RECIPES.items():
        # Join the tuple of strings into a single text block
        content = " ".join(content_tuple).lower()

        words = set(query_lower.split())
        match_count = sum(1 for word in words if word in content)
        if match_count > 0:
            results.append((match_count, doc_name, content_tuple))

    results.sort(reverse=True)
    if not results:
        return "No relevant recipes found for that query."

    # Return the top two matches
    output = []
    for _, name, content_tuple in results[:2]:
        formatted_content = "\n".join(content_tuple)
        output.append(f"[Source: {name}]\n{formatted_content}")
    return "\n\n".join(output)


# Quick test
print(search_recipes("fresh parsley"))


[Source: paneer_butter_masala]
250g paneer cut into cubes
2 medium onions roughly chopped
3 medium tomatoes chopped
2 tbsp butter
1 tbsp cooking oil
1 tbsp ginger garlic paste
10 cashew nuts
1/2 cup fresh cream
1 tsp cumin seeds
1 tsp red chili powder
1/2 tsp turmeric powder
1 tsp garam masala
1 tsp coriander powder
1 tsp kasuri methi
Salt to taste
Fresh coriander leaves for garnish
Heat oil and butter in a pan over medium heat.
Add cumin seeds and allow them to crackle.
Add chopped onions and cook until soft and translucent.
Add ginger garlic paste and cook for another minute.
Add chopped tomatoes and cashew nuts. Cook until the tomatoes become soft.
Allow the mixture to cool slightly and blend into a smooth paste.
Heat the remaining butter in the pan and add red chili powder, turmeric powder and coriander powder.
Add the prepared tomato-onion paste and cook for 5-7 minutes.
Add garam masala, salt and a little water. Mix well and bring to a gentle simmer.
Add paneer cubes and cook for

In [20]:
# @title Ask gemini to search for something
import google.generativeai as genai
import time

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

model = genai.GenerativeModel(
    model_name="gemini-3.5-flash-lite",
    tools=[search_recipes ,suggest_substitutions],
    system_instruction=(
        '''
        You are a chef training at a Michelin star restaurant.
        When you need specific information about temperatures,
        ingredients to use, cook time etc. use the search_documents tool.
        Only answer based on what you find in the documents.
        If the documents do not contain the answer, say so clearly.
        '''
    )
)

print("Model ready:", model.model_name)

Model ready: models/gemini-3.5-flash-lite


In [15]:
import time
from google import genai
from google.genai import types

client = genai.Client()  # reads GEMINI_API_KEY / GOOGLE_API_KEY from env

search_tool = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="search_recipes",
        description="Search the cookbook to find most the relevant recipe to cook.",
        parameters={
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    )
])

config = types.GenerateContentConfig(
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)


def send_with_retry(contents, max_retries=3):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model=model.model_name,
                contents=contents, config=config
            )
        except Exception as e:
            if "429" in str(e) and attempt < max_retries - 1:
                wait = 20 * (attempt + 1)
                print(f"  Rate limit hit, waiting {wait}s...")
                time.sleep(wait)
            else:
                raise


def run_agent(question: str, max_steps: int = 8) -> str:
    print(f"Question: {question}\n" + "-" * 50)
    contents = [types.Content(role="user", parts=[types.Part(text=question)])]

    for step in range(1, max_steps + 1):
        response = send_with_retry(contents)
        model_content = response.candidates[0].content

        # Append the model's turn UNMODIFIED, so thought signatures are preserved
        contents.append(model_content)

        calls = [p.function_call for p in model_content.parts if p.function_call]
        if not calls:
            print(f"\nFinal answer:\n{response.text}")
            return response.text

        # Run every requested tool call, reply with all results in ONE user turn
        result_parts = []
        for call in calls:
            args = dict(call.args)
            print(f"Step {step}: model calls {call.name}({args})")
            result = search_recipes(**args)
            print(f"  Result preview: {str(result)[:100]}...")
            result_parts.append(
                types.Part.from_function_response(
                    name=call.name, response={"result": result}
                )
            )
        contents.append(types.Content(role="user", parts=result_parts))

    return "(Stopped: hit max_steps)"


_ = run_agent("What can I cook with vanilla and caramel")

Question: What can I cook with vanilla and caramel
--------------------------------------------------
Step 1: model calls search_recipes({'query': 'vanilla caramel'})
  Result preview: [Source: chocolate_brownies]
1/2 cup unsalted butter
1 cup granulated sugar
2 large eggs
1 tsp vanil...

Final answer:
Vanilla and caramel are a classic flavor combination that work wonderfully together in all kinds of sweet treats! Here are a few delicious ways to use them:

* **Caramel Ice Cream with Vanilla:** Make a rich homemade vanilla bean ice cream and drizzle a warm, gooey caramel sauce generously over the top.
* **Salted Caramel Vanilla Cupcakes:** Bake moist vanilla cupcakes and fill or top them with a luscious salted caramel sauce, finished with vanilla buttercream frosting.
* **Caramel-Stuffed Vanilla Cookies:** Make a classic vanilla sugar or chocolate chip cookie dough, wrap portions around a soft caramel cube, and bake until gooey in the center.
* **Caramel Flan or Crème Caramel:** This c

In [17]:
questions = [
    "What is the best chicken dish on our menu?",
    "Does our restaurant's pasta use ravioli or linguine?",
    " which is the best panneer dish in restaurant?"
    "How long must the rice be cooked for",
    # This question is not answered by our database.
    "How many customers eat our food everyday?",
    # This question is part of the general knowledge of the model.
    "What is the most famous food item in Japan?"
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)  # stay within the free-tier rate limit.
    print("\n" + "=" * 60)
    run_agent(q)



Question: What is the best chicken dish on our menu?
--------------------------------------------------
Step 1: model calls search_recipes({'query': 'chicken'})
  Result preview: [Source: chicken_biryani]
500g chicken pieces
2 cups basmati rice
1 cup plain yogurt
2 large onions ...

Final answer:
Our best chicken dish on the menu is the delicious **Chicken Biryani**! It features tender chicken pieces marinated in yogurt and spices, layered with aromatic basmati rice, and cooked to perfection with fresh herbs and whole spices. 

Would you like the full recipe to make it?

Question: Does our restaurant's pasta use ravioli or linguine?
--------------------------------------------------
Step 1: model calls search_recipes({'query': 'pasta'})
  Result preview: No relevant recipes found for that query....
Step 2: model calls search_recipes({'query': 'ravioli'})
  Result preview: No relevant recipes found for that query....
Step 3: model calls search_recipes({'query': 'linguine'})
  Result pr

# Next steps:
* Save a copy of this code to your GitHub repo.
* Create a tool to recommend ingredient substitutions for recipes.
* Push the new tool to your colab and v2 to the same GitHub repo.